### Evaluating the Ideal Chunk Size for a RAG System using LlamaIndex


Introduction

Retrieval-augmented generation (RAG) has introduced an innovative approach that fuses the extensive retrieval capabilities of search systems with the LLM. When implementing a RAG system, one critical parameter that governs the system’s efficiency and performance is the chunk_size. 

How does one discern the optimal chunk size for seamless retrieval? This is where LlamaIndex Response Evaluation comes handy. In this blogpost, we'll guide you through the steps to determine the best chunk size using LlamaIndex’s Response Evaluation module. 

If you're unfamiliar with the Response Evaluation module, we recommend reviewing its documentation before proceeding.

Why Chunk Size Matters

Choosing the right chunk_size is a critical decision that can influence the efficiency and accuracy of a RAG system in several ways:

Relevance and Granularity: A small chunk_size, like 128, yields more granular chunks. 

This granularity, however, presents a risk: vital information might not be among the top retrieved chunks, especially if the similarity_top_k setting is as restrictive as 2. Conversely, a chunk size of 512 is likely to encompass all necessary information within the top chunks, ensuring that answers to queries are readily available. 

To navigate this, we employ the Faithfulness and Relevancy metrics. These measure the absence of ‘hallucinations’ and the ‘relevancy’ of responses based on the query and the retrieved contexts respectively.

Response Generation Time: As the chunk_size increases, so does the volume of information directed into the LLM to generate an answer. While this can ensure a more comprehensive context, it might also slow down the system.

Ensuring that the added depth doesn't compromise the system's responsiveness is crucial.
In essence, determining the optimal chunk_size is about striking a balance: capturing all essential information without sacrificing speed. 

It's vital to undergo thorough testing with various sizes to find a configuration that suits the specific use-case and dataset.

Setup

Before embarking on the experiment, we need to ensure all requisite modules are imported:

In [20]:
# !uv pip install llama-index pypdf

In [21]:
import nest_asyncio
from dotenv import load_dotenv
import os

load_dotenv()

nest_asyncio.apply()

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Settings,
)
from llama_index.core.evaluation import (
    DatasetGenerator,
    FaithfulnessEvaluator,
    RelevancyEvaluator
)
from llama_index.llms.openai import OpenAI
import openai
import time

openai.api_key = os.getenv('OPENAI_API_KEY') # set your openai api key

if openai.api_key:
    print("OPENAI_API_KEY:", openai.api_key[:6] + "..." + openai.api_key[-4:])
else:
    print("Warning: OPENAI_API_KEY not found in environment variables")


OPENAI_API_KEY: sk-svc...xMYA


Download Data


We'll be using the Uber 10K SEC Filings for 2021 for this experiment.

In [22]:
import os
from datetime import datetime

os.makedirs('data/10k/', exist_ok=True)



print(f"\n\n****************************************************** ")

print(f"Current working directory: {os.getcwd()}")

print(f"Current execution time is; datetime: {datetime.now()}")

print(f"****************************************************** ")



****************************************************** 
Current working directory: d:\AiCode\llama-index-tutorials
Current execution time is; datetime: 2026-01-06 10:57:32.468241
****************************************************** 


In [23]:
import urllib.request

url = 'https://raw.githubusercontent.com/jerryjliu/llama_index/main/docs/examples/data/10k/uber_2021.pdf'
output_path = 'data/10k/uber_2021.pdf'

print(f"Downloading {url}...")
urllib.request.urlretrieve(url, output_path)
print(f"Successfully downloaded to {output_path}")



print(f"\n\n****************************************************** ")

print(f"Current working directory: {os.getcwd()}")

print(f"Current execution time is; datetime: {datetime.now()}")

print(f"****************************************************** ")

Successfully downloaded to data/10k/uber_2021.pdf


****************************************************** 
Current working directory: d:\AiCode\llama-index-tutorials
Current execution time is; datetime: 2026-01-06 10:57:34.569315
****************************************************** 


Load Data

Let’s load our document.

In [24]:
# Load Data

reader = SimpleDirectoryReader("./data/10k/")
documents = reader.load_data()


print(f"\n\n****************************************************** ")

print(f"Current working directory: {os.getcwd()}")

print(f"Current execution time is; datetime: {datetime.now()}")

print(f"****************************************************** ")



****************************************************** 
Current working directory: d:\AiCode\llama-index-tutorials
Current execution time is; datetime: 2026-01-06 10:57:42.716121
****************************************************** 


In [25]:
# Inspect the loaded documents

# 1. Check how many documents were loaded
print(f"Number of documents loaded: {len(documents)}")
print("-" * 80)

# 2. Check the type of documents
print(f"Type of documents: {type(documents)}")
print(f"Type of first document: {type(documents[0])}")
print("-" * 80)

# 3. Inspect the first document
if documents:
    first_doc = documents[0]
    
    # Get the text content
    doc_text = first_doc.text
    
    # Show first 500 characters
    print("First 500 characters:")
    print(doc_text[:500])
    print("-" * 80)
    
    # Show total character count
    print(f"Total characters in first document: {len(doc_text)}")
    print("-" * 80)
    
    # Show first few sentences (split by period)
    sentences = doc_text.split('.')
    print("First 3 sentences:")
    for i, sentence in enumerate(sentences[:3], 1):
        print(f"{i}. {sentence.strip()}")
    print("-" * 80)
    
    # Show metadata if available
    print("Document metadata:")
    print(first_doc.metadata)
    print("-" * 80)
    
    # Show word count
    word_count = len(doc_text.split())
    print(f"Word count in first document: {word_count}")
    print("-" * 80)

# 4. Summary of all documents
print("\nSummary of all documents:")
for i, doc in enumerate(documents):
    print(f"Document {i+1}: {len(doc.text)} characters, {len(doc.text.split())} words")



print(f"\n\n****************************************************** ")

print(f"Current working directory: {os.getcwd()}")

print(f"Current execution time is; datetime: {datetime.now()}")

print(f"****************************************************** ")


Number of documents loaded: 307
--------------------------------------------------------------------------------
Type of documents: <class 'list'>
Type of first document: <class 'llama_index.core.schema.Document'>
--------------------------------------------------------------------------------
First 500 characters:
UNITED STATES
SECURITIES AND EXCHANGE COMMISSIONWashington, D.C. 20549 ____________________________________________ FORM 10-K____________________________________________ (Mark One)☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934For the fiscal year ended  December 31, 2021OR☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934For the transition period from_____ to _____            Commission File Number: 001-38902_________
--------------------------------------------------------------------------------
Total characters in first document: 2568
------------------------------------------------------------

Question Generation

To select the right chunk_size, we'll compute metrics like Average Response time, Faithfulness, and Relevancy for various chunk_sizes. 

The DatasetGenerator will help us generate questions from the documents.


In [26]:
# To evaluate for each chunk size, we will first generate a set of 40 questions from first 20 pages.
eval_documents = documents[:20]
data_generator = DatasetGenerator.from_documents(eval_documents)
eval_questions = data_generator.generate_questions_from_nodes(num = 40)



print(f"\n\n****************************************************** ")

print(f"Current working directory: {os.getcwd()}")

print(f"Current execution time is; datetime: {datetime.now()}")

print(f"****************************************************** ")

d:\AiCode\llama-index-tutorials\llmai_venv\Lib\site-packages\llama_index\core\evaluation\dataset_generation.py:201: DeprecationWarning: Call to deprecated class DatasetGenerator. (Deprecated in favor of `RagDatasetGenerator` which should be used instead.)
  return cls(
2026-01-06 10:57:54,422 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 10:57:54,467 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 10:57:54,469 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 10:57:54,504 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 10:57:54,554 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 10:57:54,641 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 10:57:54,649 - INFO - HTTP Request: POST 



****************************************************** 
Current working directory: d:\AiCode\llama-index-tutorials
Current execution time is; datetime: 2026-01-06 10:57:56.837851
****************************************************** 


d:\AiCode\llama-index-tutorials\llmai_venv\Lib\site-packages\llama_index\core\evaluation\dataset_generation.py:297: DeprecationWarning: Call to deprecated class QueryResponseDataset. (Deprecated in favor of `LabelledRagDataset` which should be used instead.)
  return QueryResponseDataset(queries=queries, responses=responses_dict)


Setting Up Evaluators
We are setting up the GPT-4 model to serve as the backbone for evaluating the responses generated during the experiment. Two evaluators, FaithfulnessEvaluator and RelevancyEvaluator, are initialised with the service_context .

Faithfulness Evaluator - It is useful for measuring if the response was hallucinated and measures if the response from a query engine matches any source nodes.
Relevancy Evaluator - It is useful for measuring if the query was actually answered by the response and measures if the response + source nodes match the query.

In [27]:
# We will use GPT-4 for evaluating the responses
gpt4 = OpenAI(temperature=0, model="gpt-4o-mini")

# Define Faithfulness and Relevancy Evaluators which are based on GPT-4
## service_context_gpt4 = ServiceContext.from_defaults(llm=gpt4) 
## faithfulness_gpt4 = FaithfulnessEvaluator(service_context=service_context_gpt4)  This Servicecontext has been deprecated in latest version of llama-index

faithfulness_gpt4 = FaithfulnessEvaluator(llm=gpt4)

## relevancy_gpt4 = RelevancyEvaluator(service_context=service_context_gpt4) This Servicecontext has been deprecated in latest version of llama-index

relevancy_gpt4 = RelevancyEvaluator(llm=gpt4)


Response Evaluation For A Chunk Size

We evaluate each chunk_size based on 3 metrics.

Average Response Time.

Average Faithfulness.

Average Relevancy.

Here's a function, evaluate_response_time_and_accuracy, that does just that which has:

VectorIndex Creation.

Building the Query Engine**.**

Metrics Calculation.

In [28]:
# Define function to calculate average response time, average faithfulness and average relevancy metrics for given chunk size
# We use GPT-3.5-Turbo to generate response and GPT-4 to evaluate it.

def evaluate_response_time_and_accuracy(chunk_size, eval_questions):
    """
    Evaluate the average response time, faithfulness, and relevancy of responses generated by GPT-3.5-turbo for a given chunk size.

    Parameters:
    chunk_size (int): The size of data chunks being processed.
    eval_questions (list): A list of questions to evaluate the responses.

    Returns:
    tuple: A tuple containing the average response time, faithfulness, and relevancy metrics.
    """

    total_response_time = 0
    total_faithfulness = 0
    total_relevancy = 0

    # create vector index
    llm = OpenAI(model="gpt-3.5-turbo")
    
    '''  service_context has been deprecated in latest version of llama-index
    use the Settings object to set llm and chunk_size globally.

    service_context = ServiceContext.from_defaults(llm=llm, chunk_size=chunk_size)
    vector_index = VectorStoreIndex.from_documents(
        eval_documents, service_context=service_context
    )
    '''
    # Configure Settings with the chunk size
    Settings.llm = llm
    Settings.chunk_size = chunk_size
    
    vector_index = VectorStoreIndex.from_documents(
        eval_documents
    )

    # build query engine
    # By default, similarity_top_k is set to 2. To experiment with different values, pass it as an argument to as_query_engine()

    query_engine = vector_index.as_query_engine()
    num_questions = len(eval_questions)

    # Iterate over each question in eval_questions to compute metrics.
    # While BatchEvalRunner can be used for faster evaluations (see: https://docs.llamaindex.ai/en/latest/examples/evaluation/batch_eval.html),
    # we're using a loop here to specifically measure response time for different chunk sizes.

    for question in eval_questions:
        start_time = time.time()
        response_vector = query_engine.query(question)
        elapsed_time = time.time() - start_time
        
        if response_vector is not None:
            # Keep the original response object for evaluation
            faithfulness_result = faithfulness_gpt4.evaluate_response(response=response_vector).passing
            relevancy_result = relevancy_gpt4.evaluate_response(query=question, response=response_vector).passing
            
            total_response_time += elapsed_time
            total_faithfulness += faithfulness_result
            total_relevancy += relevancy_result

    average_response_time = total_response_time / num_questions
    average_faithfulness = total_faithfulness / num_questions
    average_relevancy = total_relevancy / num_questions

    return average_response_time, average_faithfulness, average_relevancy

Testing Across Different Chunk Sizes
We'll evaluate a range of chunk sizes to identify which offers the most promising metrics

In [ ]:
# Iterate over different chunk sizes to evaluate the metrics and store results

# Initialize dictionary to store results for each chunk size
chunk_size_results = {}

for chunk_size in [128, 256, 512, 1024, 2048]:
    avg_response_time, avg_faithfulness, avg_relevancy = evaluate_response_time_and_accuracy(chunk_size, eval_questions)
    
    # Store results in dictionary
    chunk_size_results[chunk_size] = {
        'average_response_time': avg_response_time,
        'average_faithfulness': avg_faithfulness,
        'average_relevancy': avg_relevancy
    }
    
    # Print progress
    print(f"Chunk size {chunk_size} - Average Response time: {avg_response_time:.2f}s, Average Faithfulness: {avg_faithfulness:.2f}, Average Relevancy: {avg_relevancy:.2f}")

print("\n" + "="*80)
print("Evaluation complete! Results stored in 'chunk_size_results' dictionary.")
print("="*80)


# 9 minutes 16.6 Seconds.. For Next Iteration please reduce the 307 pages PDF 

2026-01-06 11:24:23,658 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:24:24,335 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:24:25,046 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:24:25,429 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:24:26,102 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:24:26,663 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:24:27,174 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:24:28,091 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:24:28,100 - INFO - Retrying request to /chat/completions in 0.449936 seconds
2026-01-06 11:24:29,058 - INFO - HTTP Request: 

Chunk size 128 - Average Response time: 0.80s, Average Faithfulness: 0.60, Average Relevancy: 0.68


2026-01-06 11:26:15,451 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:26:16,363 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:26:17,052 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:26:17,272 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:26:18,072 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:26:18,082 - INFO - Retrying request to /chat/completions in 0.476451 seconds
2026-01-06 11:26:18,910 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:26:18,919 - INFO - Retrying request to /chat/completions in 0.386945 seconds
2026-01-06 11:26:19,743 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:26:19,894 - INFO - HTTP Request: POST

Chunk size 256 - Average Response time: 0.80s, Average Faithfulness: 0.62, Average Relevancy: 0.75


2026-01-06 11:28:05,044 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:28:05,219 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:28:06,152 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:28:06,161 - INFO - Retrying request to /chat/completions in 0.384986 seconds
2026-01-06 11:28:06,994 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:28:07,004 - INFO - Retrying request to /chat/completions in 0.471205 seconds
2026-01-06 11:28:08,842 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:28:09,007 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:28:09,606 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:28:09,620 - INFO - Retrying req

Chunk size 512 - Average Response time: 0.75s, Average Faithfulness: 0.65, Average Relevancy: 0.70


2026-01-06 11:29:52,849 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:29:53,061 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:29:54,259 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:29:54,270 - INFO - Retrying request to /chat/completions in 0.430870 seconds
2026-01-06 11:29:55,039 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:29:55,060 - INFO - Retrying request to /chat/completions in 0.459942 seconds
2026-01-06 11:29:57,108 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:29:57,319 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:29:57,768 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:29:57,775 - INFO - Retrying req

Chunk size 1024 - Average Response time: 0.77s, Average Faithfulness: 0.68, Average Relevancy: 0.80


2026-01-06 11:31:42,569 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:31:42,694 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:31:43,479 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:31:43,488 - INFO - Retrying request to /chat/completions in 0.463337 seconds
2026-01-06 11:31:44,430 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:31:44,449 - INFO - Retrying request to /chat/completions in 0.490427 seconds
2026-01-06 11:31:45,383 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:31:45,568 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-01-06 11:31:45,994 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-01-06 11:31:46,007 - INFO - Retrying req

Chunk size 2048 - Average Response time: 0.68s, Average Faithfulness: 0.70, Average Relevancy: 0.85

Evaluation complete! Results stored in 'chunk_size_results' dictionary.


In [31]:
# Display the stored results in a structured format

import pandas as pd

# Convert dictionary to DataFrame for better visualization
df_results = pd.DataFrame.from_dict(chunk_size_results, orient='index')
df_results.index.name = 'chunk_size'
df_results = df_results.reset_index()

print("="*80)
print("CHUNK SIZE EVALUATION RESULTS")
print("="*80)
print(df_results.to_string(index=False))
print("="*80)

# Display the raw dictionary
print("\nRaw dictionary format:")
print("-"*80)
for chunk_size, metrics in chunk_size_results.items():
    print(f"\nChunk Size: {chunk_size}")
    print(f"  Average Response Time: {metrics['average_response_time']:.4f}s")
    print(f"  Average Faithfulness: {metrics['average_faithfulness']:.4f}")
    print(f"  Average Relevancy: {metrics['average_relevancy']:.4f}")

# Find the best chunk size based on different criteria
print("\n" + "="*80)
print("OPTIMAL CHUNK SIZE ANALYSIS")
print("="*80)

best_response_time = min(chunk_size_results.items(), key=lambda x: x[1]['average_response_time'])
best_faithfulness = max(chunk_size_results.items(), key=lambda x: x[1]['average_faithfulness'])
best_relevancy = max(chunk_size_results.items(), key=lambda x: x[1]['average_relevancy'])

print(f"\nBest Response Time: Chunk size {best_response_time[0]} ({best_response_time[1]['average_response_time']:.4f}s)")
print(f"Best Faithfulness: Chunk size {best_faithfulness[0]} ({best_faithfulness[1]['average_faithfulness']:.4f})")
print(f"Best Relevancy: Chunk size {best_relevancy[0]} ({best_relevancy[1]['average_relevancy']:.4f})")

# Calculate a balanced score (you can adjust weights as needed)
print("\n" + "-"*80)
print("Balanced Score (considering all metrics equally):")
print("-"*80)
for chunk_size, metrics in chunk_size_results.items():
    # Normalize metrics (lower response time is better, higher faithfulness and relevancy are better)
    # For simplicity, we'll use a weighted average
    # Note: We invert response time so higher is better
    max_response_time = max([m['average_response_time'] for m in chunk_size_results.values()])
    normalized_response_time = 1 - (metrics['average_response_time'] / max_response_time)
    
    balanced_score = (normalized_response_time + metrics['average_faithfulness'] + metrics['average_relevancy']) / 3
    print(f"Chunk size {chunk_size}: Balanced Score = {balanced_score:.4f}")

CHUNK SIZE EVALUATION RESULTS
 chunk_size  average_response_time  average_faithfulness  average_relevancy
        128               0.795760                 0.600              0.675
        256               0.799475                 0.625              0.750
        512               0.748156                 0.650              0.700
       1024               0.771940                 0.675              0.800
       2048               0.682330                 0.700              0.850

Raw dictionary format:
--------------------------------------------------------------------------------

Chunk Size: 128
  Average Response Time: 0.7958s
  Average Faithfulness: 0.6000
  Average Relevancy: 0.6750

Chunk Size: 256
  Average Response Time: 0.7995s
  Average Faithfulness: 0.6250
  Average Relevancy: 0.7500

Chunk Size: 512
  Average Response Time: 0.7482s
  Average Faithfulness: 0.6500
  Average Relevancy: 0.7000

Chunk Size: 1024
  Average Response Time: 0.7719s
  Average Faithfulness: 0.6750
